# 08 — EGFR 共有結合ドッキングチュートリアル
# EGFR Covalent Docking Tutorial

**ターゲット**: EGFR キナーゼドメイン（Cys797 共有結合ポケット）  
**事例**: Afatinib (2nd gen / 不可逆, 4G5J) vs Osimertinib (3rd gen / T790M 耐性克服, 6LUD)  
**ウォーヘッド**: Acrylamide（Michael acceptor → Cys797 SG）

---

## EGFR 阻害剤の世代進化

| 世代 | 代表薬 | 結合様式 | 課題 |
|------|-------|---------|------|
| 1st gen | Erlotinib (1XKK) | 可逆 / Type I | 耐性 (T790M) |
| 2nd gen | **Afatinib (4G5J)** | **不可逆 / Cys797 共有結合** | HER1-4 汎阻害（毒性） |
| 3rd gen | **Osimertinib (6LUD)** | **不可逆 / T790M 克服** | C797S 耐性 |

→ Issue #75 (erlotinib / lapatinib) の続編として、共有結合ドッキングを初実戦投入します。

---

## このノートブックで学べること

1. 共有結合ドッキングの概念とウォーヘッド構造の確認方法
2. PDB から Cys797 座標・残基番号を自動検出する手順
3. `UniDock2Runner(UniDock2RunConfig(covalent_docking=True, ...))` の設定
4. `compute_strain_energy()` と `StrainEnergyFilter` で共有結合ポーズの品質評価
5. `StructuralAlertFilter` でウォーヘッドの反応性スクリーニング
6. 2nd gen vs 3rd gen の結合様式・T790M 耐性機序の比較

> **Note**: ドッキング実行セクション（Section 5）は UniDock2 バイナリと GPU が必要です。  
> バイナリなしでも Section 6 以降の解析デモは結晶構造ポーズで実行可能です。

In [ ]:
# CONFIG -----------------------------------------------------------------------
DATA_DIR       = "../data/egfr_covalent"   # PDB ダウンロード先
RESULTS_DIR    = "../results/egfr_covalent" # ドッキング結果出力先
UNIDOCK2_BIN   = "unidock2"                # path to UniDock2 binary
# ------------------------------------------------------------------------------

## 1. セットアップ / Setup

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from IPython.display import display

# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis import DockingResult, get_reader
from docking_analysis.preparation.receptor import load_receptor, prepare_receptor
from docking_analysis.preparation.gridbox import gridbox_from_ligand
from mdatools.docking.fingerprints.prolif import ProLIFCalculator
from mdatools.docking.visualization.interaction_map import draw_interaction_map
from mdatools.docking.analysis.properties import calculate_properties
from mdatools.docking.analysis.strain import compute_strain_energy, add_strain_to_df
from mdatools.docking.selection.filters import (
    StrainEnergyFilter, StructuralAlertFilter, apply_filters
)
from mdatools.docking.clustering.chemical import cluster_by_butina

RDLogger.DisableLog("rdApp.warning")

data_dir    = Path(DATA_DIR)
results_dir = Path(RESULTS_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
print("Setup complete.")

## 2. PDB 構造の取得 / Download PDB Structures

| PDB ID | リガンド | 世代 | 分解能 | 特徴 |
|--------|---------|------|--------|-----|
| **4G5J** | Afatinib | 2nd gen (不可逆) | 1.9 Å | アクリルアミド → Cys797 共有結合。非常に高分解能 |
| **6LUD** | Osimertinib | 3rd gen (T790M 克服) | 2.1 Å | インドールアクリルアミド → Cys797。T790M 残基 Met790 との接触 |

In [ ]:
PDB_IDS = ["4G5J", "6LUD"]

for pdb_id in PDB_IDS:
    dest = data_dir / f"{pdb_id.lower()}.pdb"
    if not dest.exists() or dest.stat().st_size == 0:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Downloading {pdb_id}...", end=" ")
        urllib.request.urlretrieve(url, dest)
        print(f"→ {dest}")
    else:
        print(f"{pdb_id}: already exists ({dest})")

## 3. リガンド・Cys797 の確認 / Identify Ligand and Cys797

共有結合ドッキングには**結合対象の残基情報**が必要です。  
UniDock2 の `covalent_residue_atom_info` に以下を渡します:

```python
covalent_residue_atom_info = [["CYS", 797, "SG"]]  # resname, resid, atom_name
```

以下のセルで Cys797 の存在・座標を確認します。

In [ ]:
def list_hetatm_residues(pdb_path: Path, min_atoms: int = 6) -> list[dict]:
    """Parse HETATM records, return large non-water residues."""
    WATER_CODES = {"HOH", "WAT", "H2O", "DOD", "D2O"}
    seen = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("HETATM"):
                continue
            res_name = line[17:20].strip()
            chain    = line[21].strip()
            seq_id   = line[22:26].strip()
            key = (chain, res_name, seq_id)
            if res_name not in WATER_CODES:
                seen[key] = seen.get(key, 0) + 1
    return sorted(
        [{"chain": k[0], "residue": k[1], "seqid": k[2], "n_atoms": v}
         for k, v in seen.items() if v >= min_atoms],
        key=lambda x: -x["n_atoms"]
    )


def find_cys_sg(pdb_path: Path, cys_resid: int = 797, chain: str = "A") -> dict | None:
    """Locate a cysteine SG atom in the PDB ATOM records.

    Returns
    -------
    dict with keys: resname, resid, chain, coords (x, y, z)
    or None if not found.
    """
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            atom_name   = line[12:16].strip()
            res_name    = line[17:20].strip()
            rec_chain   = line[21].strip()
            try:
                rec_resid = int(line[22:26].strip())
            except ValueError:
                continue
            if (res_name == "CYS" and rec_resid == cys_resid
                    and atom_name == "SG"
                    and (chain == "*" or rec_chain == chain)):
                return {
                    "resname": "CYS",
                    "resid": rec_resid,
                    "chain": rec_chain,
                    "atom": "SG",
                    "coords": (float(line[30:38]), float(line[38:46]), float(line[46:54])),
                }
    return None


# HETATM 確認
for pdb_id in PDB_IDS:
    residues = list_hetatm_residues(data_dir / f"{pdb_id.lower()}.pdb")
    print(f"\n{pdb_id} — 主要 HETATM 残基 (≥6 heavy atoms):")
    for r in residues:
        print(f"  Chain {r['chain']}: {r['residue']:>4}  seqid={r['seqid']:>5}  atoms={r['n_atoms']}")

# Cys797 SG の確認
# PDB によって残基番号が異なる場合があるため、PDB ごとにマッピングを定義
# 4G5J: canonical EGFR 番号 (797)
# 6LUD: PDB 内部番号 (775 ≡ canonical 797; EGFR 配列の欠失断片が違うため)
CYS_RESID = {
    "4G5J": 797,
    "6LUD": 775,
}

print()
cys_info = {}
for pdb_id in PDB_IDS:
    resid = CYS_RESID.get(pdb_id, 797)
    info = find_cys_sg(data_dir / f"{pdb_id.lower()}.pdb", cys_resid=resid)
    if info:
        cys_info[pdb_id] = info
        print(f"{pdb_id}: Cys797 SG found (PDB resid {resid}) — Chain {info['chain']}, "
              f"coords = ({info['coords'][0]:.2f}, {info['coords'][1]:.2f}, {info['coords'][2]:.2f})")
    else:
        print(f"{pdb_id}: Cys797 SG NOT found at resid {resid} — check residue numbering in PDB")

### 3.1 リガンドコードの設定

上の出力を確認して `LIGAND_CODES` を設定してください。

- **4G5J** (Afatinib): `"AFT"` が一般的なコード
- **6LUD** (Osimertinib): `"OSS"` または上の出力を参照

PDB の HETATM 出力で最も heavy atoms が多いものが目的リガンドです。

In [ ]:
# ↓ 上のセルの出力を見て残基コードを設定
LIGAND_CODES = {
    "4G5J": "0WN",   # afatinib (RCSB code)
    "6LUD": "YY3",   # osimertinib (RCSB code)
}

## 4. リガンド抽出・ウォーヘッドの確認
## Ligand Extraction and Warhead Verification

共有結合阻害剤のウォーヘッド（反応基）を SMARTS で確認します。

| 阻害剤 | ウォーヘッド | SMARTS |
|-------|------------|--------|
| Afatinib | Acrylamide (cis) | `C=CC(=O)N` |
| Osimertinib | Acrylamide (indole) | `C=CC(=O)N` |
| 参考: Ibrutinib | Acrylamide | `C=CC(=O)N` |

**Michael 付加反応**: `Cys797-SH + CH2=CH-C(=O)-NHR → Cys797-S-CH2-CH2-C(=O)-NHR`

In [ ]:
def extract_ligand_mol(pdb_path: Path, res_code: str, chain: str = "A") -> Chem.Mol | None:
    """Extract a ligand from PDB HETATM records and return an RDKit Mol."""
    with open(pdb_path) as f:
        lines = f.readlines()

    hetatm = [l for l in lines
               if l.startswith("HETATM") and l[17:20].strip() == res_code
               and (chain == "*" or l[21].strip() == chain)]
    if not hetatm:
        # Try any chain
        hetatm = [l for l in lines
                   if l.startswith("HETATM") and l[17:20].strip() == res_code]
    if not hetatm:
        print(f"  WARNING: residue {res_code} not found in {pdb_path.name}")
        return None

    pdb_block = "".join(hetatm) + "END\n"
    mol = Chem.MolFromPDBBlock(pdb_block, removeHs=True, sanitize=True)
    if mol is None:
        print(f"  WARNING: RDKit could not parse {res_code}")
    return mol


# Michael acceptor SMARTS (acrylamide / acrylate)
WARHEAD_SMARTS = "[CH2]=[CH]-C(=O)-[NH]"
warhead_pattern = Chem.MolFromSmarts(WARHEAD_SMARTS)

ligand_mols = {}
for pdb_id, res_code in LIGAND_CODES.items():
    mol = extract_ligand_mol(data_dir / f"{pdb_id.lower()}.pdb", res_code)
    if mol is None:
        continue
    mol.SetProp("mol_name", res_code)
    mol.SetProp("pdb_id", pdb_id)
    mol.SetProp("pose_rank", "1")
    mol.SetProp("docking_score", "0.0")
    ligand_mols[pdb_id] = mol

    # ウォーヘッド確認 (共有結合後の構造からは検出できない場合あり)
    has_warhead = mol.HasSubstructMatch(warhead_pattern)
    gen = "2nd gen" if pdb_id == "4G5J" else "3rd gen"
    print(f"{pdb_id} [{res_code}] {gen}: {mol.GetNumAtoms()} heavy atoms, "
          f"acrylamide warhead = {'✓' if has_warhead else '(not detected — may be covalently linked in crystal)'}")

# 構造表示
if ligand_mols:
    mols_list = list(ligand_mols.values())
    legends = [
        f"{pid}\n{LIGAND_CODES[pid]}\n{'Afatinib (2nd gen)' if pid == '4G5J' else 'Osimertinib (3rd gen)'}"
        for pid in ligand_mols
    ]
    img = Draw.MolsToGridImage(mols_list, molsPerRow=2, subImgSize=(400, 300), legends=legends)
    display(img)

### 4.1 共有結合リガンドの注意点

> ⚠️ **結晶構造中のリガンドは既に共有結合状態**です。
> そのため:
> - HETATM から抽出したリガンドは SG との結合が切断されたフラグメントになる場合があります
> - ウォーヘッドの C=C 二重結合が C-C 単結合になっている（Cys との付加体）
>
> ドッキング入力 SDF には**反応前（C=C ウォーヘッドを持つ）構造**を使用してください。  
> 推奨: ChEMBL や PubChem からオリジナル SMILES を取得して 3D 座標を生成する。

以下のセルで参照 SMILES から「反応前」構造を生成します。

In [ ]:
# 反応前（covalent docking に使用する）構造を SMILES から生成
# Afatinib: ChEMBL 481514
AFATINIB_SMILES  = "CC=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCC1CCN(C)CC1"
# Osimertinib: ChEMBL 3353410 (AZD9291)
OSIMERTINIB_SMILES = "COc1cc2ncnc(Nc3cccc(NC(=O)C=C)c3)c2cc1NC(=O)C=C"

pre_reaction_smiles = {
    "4G5J": AFATINIB_SMILES,
    "6LUD": OSIMERTINIB_SMILES,
}

pre_reaction_mols = {}
for pdb_id, smi in pre_reaction_smiles.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"{pdb_id}: SMILES parse failed")
        continue
    # 3D 座標生成
    mol = Chem.AddHs(mol)
    res = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    if res == -1:
        print(f"{pdb_id}: 3D embedding failed")
        continue
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("mol_name", LIGAND_CODES.get(pdb_id, pdb_id))
    pre_reaction_mols[pdb_id] = mol

    # ウォーヘッド確認
    mol_no_h = Chem.RemoveHs(mol)
    has_warhead = mol_no_h.HasSubstructMatch(warhead_pattern)
    print(f"{pdb_id}: {mol.GetNumAtoms()} atoms (with H), acrylamide warhead = {'✓' if has_warhead else '✗'}")

# 並べて表示
mols_display = [Chem.RemoveHs(m) for m in pre_reaction_mols.values()]
leg = [f"{'Afatinib' if pid == '4G5J' else 'Osimertinib'}\n({pid})" for pid in pre_reaction_mols]
img = Draw.MolsToGridImage(mols_display, molsPerRow=2, subImgSize=(400, 300), legends=leg)
display(img)

## 5. 受容体準備 / Receptor Preparation

In [ ]:
receptor_paths = {}
receptor_mols  = {}

for pdb_id in PDB_IDS:
    raw_pdb = data_dir / f"{pdb_id.lower()}.pdb"
    out_pdb = data_dir / f"{pdb_id.lower()}_receptor.pdb"

    if not out_pdb.exists():
        print(f"Preparing {pdb_id}...", end=" ")
        _, rec_mol = prepare_receptor(raw_pdb, out_pdb, protein_only=True)
        print(f"→ {out_pdb}")
    else:
        print(f"{pdb_id} receptor ready: {out_pdb}")
        rec_mol = load_receptor(out_pdb)

    receptor_paths[pdb_id] = out_pdb
    receptor_mols[pdb_id]  = rec_mol

## 5.1 グリッドボックスの設定 / Grid Box Setup

Cys797 は ATP 結合ポケットの深部にあります。  
結晶リガンドのみならず Cys797 SG 座標もグリッドボックスに含まれるよう設定します。

In [ ]:
gridboxes = {}
for pdb_id, mol in ligand_mols.items():
    if mol.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())

    # 共有結合ドッキングのため padding を少し大きめに
    grid = gridbox_from_ligand(mol, padding=5.0)
    gridboxes[pdb_id] = grid

    gen = "Afatinib / 2nd gen" if pdb_id == "4G5J" else "Osimertinib / 3rd gen"
    print(f"\n{pdb_id} [{gen}]:")
    print(f"  Grid center (Å): ({grid.center[0]:.2f}, {grid.center[1]:.2f}, {grid.center[2]:.2f})")
    print(f"  Grid size   (Å): ({grid.size[0]:.2f}, {grid.size[1]:.2f}, {grid.size[2]:.2f})")

    # Cys797 SG がグリッドボックス内に入っているか確認
    if pdb_id in cys_info:
        cx, cy, cz = cys_info[pdb_id]["coords"]
        gx, gy, gz = grid.center
        sx, sy, sz = grid.size[0]/2, grid.size[1]/2, grid.size[2]/2
        in_box = (abs(cx - gx) <= sx and abs(cy - gy) <= sy and abs(cz - gz) <= sz)
        print(f"  Cys797 SG in grid box: {'✓' if in_box else '⚠ outside — consider increasing padding'}")

## 6. UniDock2 共有結合ドッキングの設定
## UniDock2 Covalent Docking Configuration

> ⚠️ このセクションは UniDock2 バイナリと GPU が必要です。  
> バイナリがない場合は Section 7（結晶構造ポーズ解析デモ）に進んでください。

**UniDock2 covalent docking の設定:**
```yaml
Preprocessing:
    covalent_ligand: true
    covalent_residue_atom_info_list: [["CYS", 797, "SG"]]
```

**動作原理:**  
UniDock2 は `covalent_residue_atom_info_list` で指定された原子を固定アンカーとして扱い、  
ウォーヘッド原子との距離拘束付きでリガンドをサンプリングします。

In [ ]:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.docking import get_runner
from docking_analysis.docking.unidock2 import UniDock2Runner, UniDock2RunConfig

# 共有結合ドッキング設定
# covalent_residue_atom_info format: [[resname, resid, atom_name], ...]
cov_config = UniDock2RunConfig(
    unidock2_binary=UNIDOCK2_BIN,
    covalent_docking=True,
    covalent_residue_atom_info=[["CYS", 797, "SG"]],
    exhaustiveness=512,
    num_pose=10,
    task="screen",
)

print("UniDock2RunConfig (covalent) — YAML preview:")
runner = UniDock2Runner(cov_config)
if gridboxes:
    sample_grid = next(iter(gridboxes.values()))
    sample_yaml = runner._build_yaml_config(
        receptor=data_dir / "4g5j_receptor.pdb",
        ligand=data_dir / "afatinib.sdf",
        grid=sample_grid,
        output_sdf=results_dir / "4g5j_afatinib_out.sdf",
    )
    print(sample_yaml)

In [ ]:
# ドッキング入力 SDF を保存（反応前構造を使用）
DRUG_NAMES = {"4G5J": "afatinib", "6LUD": "osimertinib"}
input_sdf_paths = {}

for pdb_id, mol in pre_reaction_mols.items():
    sdf_path = data_dir / f"{DRUG_NAMES[pdb_id]}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    writer.write(mol)
    writer.close()
    input_sdf_paths[pdb_id] = sdf_path
    print(f"Saved: {sdf_path}")

In [ ]:
# ドッキング実行 (UniDock2 + GPU が必要)
import shutil

docking_results = {}

if shutil.which(UNIDOCK2_BIN) is None:
    print(f"⚠ '{UNIDOCK2_BIN}' not found in PATH — skipping docking.")
    print("GPU 環境でのドッキング手順:")
    print("  docker compose -f docker/docker-compose.yml --profile gpu up")
    print("Section 7 uses crystal structure poses as demo data instead.")
else:
    for pdb_id in PDB_IDS:
        if pdb_id not in input_sdf_paths or pdb_id not in gridboxes:
            continue

        out_sdf = results_dir / f"{pdb_id.lower()}_{DRUG_NAMES[pdb_id]}_cov_out.sdf"
        print(f"\nRunning covalent docking: {pdb_id}...")
        try:
            result = runner.run(
                ligand=input_sdf_paths[pdb_id],
                receptor=receptor_paths[pdb_id],
                grid=gridboxes[pdb_id],
                output=out_sdf,
            )
            docking_results[pdb_id] = result
            print(f"  Top score : {result.scores[0]:.2f} kcal/mol")
            print(f"  Poses     : {len(result.poses)}")
        except RuntimeError as e:
            print(f"  Docking failed: {e}")

## 7. 結合様式の解析（結晶構造ポーズ使用）
## Binding Mode Analysis (Using Crystal Poses)

ドッキング結果がない場合は、結晶構造から抽出したポーズで解析デモを実行します。

In [ ]:
# ドッキング結果がなければ結晶構造を使用
analysis_results = {}
for pdb_id, mol in ligand_mols.items():
    if pdb_id in docking_results:
        analysis_results[pdb_id] = docking_results[pdb_id]
        print(f"{pdb_id}: using docking result ({len(docking_results[pdb_id].poses)} poses)")
    else:
        analysis_results[pdb_id] = DockingResult(
            poses=[mol],
            scores=[0.0],
            source_file=data_dir / f"{pdb_id.lower()}.pdb",
            backend="crystal",
        )
        print(f"{pdb_id}: using crystal structure pose")

### 7.1 ProLIF 相互作用フィンガープリント
### ProLIF Interaction Fingerprints

**EGFR 共有結合阻害剤の主要な結合残基:**

| 残基 | 役割 | Afatinib | Osimertinib |
|------|-----|---------|-------------|
| **Cys797** | 共有結合形成 | ✓ (SG) | ✓ (SG) |
| **Met793** | ヒンジ H 結合 | ✓ | ✓ |
| **Thr790 (T790M)** | 耐性変異部位 | △ (野生型) | ✓ (Met790 接触) |
| **Lys745** | 触媒 Lys | 疎水接触 | 疎水接触 |
| **Asp855** | DFG モチーフ | − | − |

In [ ]:
calculator = ProLIFCalculator()

fp_results = {}
for pdb_id, result in analysis_results.items():
    rec_mol = receptor_mols.get(pdb_id)
    if rec_mol is None:
        print(f"  {pdb_id}: receptor not loaded — skip")
        continue

    print(f"\n{pdb_id} — ProLIF fingerprint...")
    fp_df = calculator.calculate(result.poses, rec_mol, show_progress=True)
    fp_results[pdb_id] = fp_df

    active_cols = fp_df.columns[fp_df.any()].tolist()
    print(f"  検出された相互作用 ({len(active_cols)} 件):")
    for col in active_cols:
        print(f"    {col}")

### 7.2 相互作用の比較 (2nd gen vs 3rd gen)
### Interaction Comparison (2nd gen vs 3rd gen)

In [ ]:
if len(fp_results) == 2:
    cols_4g5j = set(fp_results["4G5J"].columns[fp_results["4G5J"].any()])
    cols_6lud = set(fp_results["6LUD"].columns[fp_results["6LUD"].any()])

    common    = cols_4g5j & cols_6lud
    only_4g5j = cols_4g5j - cols_6lud
    only_6lud = cols_6lud - cols_4g5j

    print(f"共通の相互作用 ({len(common)} 件):")
    for c in sorted(common):    print(f"  {c}")

    print(f"\n4G5J のみ (Afatinib / 2nd gen, {len(only_4g5j)} 件):")
    for c in sorted(only_4g5j): print(f"  {c}")

    print(f"\n6LUD のみ (Osimertinib / 3rd gen, {len(only_6lud)} 件):")
    for c in sorted(only_6lud): print(f"  {c}")

### 7.3 インタラクションマップの比較
### Interaction Map Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

pdb_list = ["4G5J", "6LUD"]
titles = {
    "4G5J": "Afatinib (2nd gen)\nCys797 covalent / irreversible",
    "6LUD": "Osimertinib (3rd gen)\nT790M resistance / Cys797 covalent",
}

for ax, pdb_id in zip(axes, pdb_list):
    if pdb_id not in fp_results:
        ax.set_title(f"{pdb_id}: data not available")
        continue

    fp_df      = fp_results[pdb_id]
    mol        = ligand_mols[pdb_id]
    active_cols = fp_df.columns[fp_df.any()].tolist()

    if active_cols:
        draw_interaction_map(mol=mol, interactions=active_cols,
                             ax=ax, title=f"{pdb_id}\n{titles[pdb_id]}")
    else:
        ax.set_title(f"{pdb_id}: no interactions detected")

plt.tight_layout()
out_fig = results_dir / "egfr_covalent_interaction_map.png"
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig}")

## 8. ポーズ品質評価 — 歪みエネルギーとフィルタリング
## Pose Quality Assessment — Strain Energy and Filtering

共有結合ドッキングポーズは非共有結合ポーズより歪みが大きくなりやすいです。  
`compute_strain_energy()` で品質をスクリーニングします。

**推奨閾値:**
- `strain_energy < 5 kcal/mol` — 優良なポーズ
- `strain_energy < 10 kcal/mol` — 許容範囲
- `strain_energy ≥ 10 kcal/mol` — 要注意（非現実的な配座の可能性）

In [ ]:
from mdatools.docking.analysis.strain import compute_strain_energy, add_strain_to_df

strain_rows = []
for pdb_id, result in analysis_results.items():
    for i, mol in enumerate(result.poses):
        # 水素を追加してから歪みエネルギーを計算
        mol_h = Chem.AddHs(mol)
        if mol_h.GetNumConformers() == 0:
            AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3())
        strain = compute_strain_energy(mol_h)
        strain_rows.append({
            "pdb_id": pdb_id,
            "ligand": LIGAND_CODES.get(pdb_id, pdb_id),
            "pose_rank": i + 1,
            "docking_score": result.scores[i] if i < len(result.scores) else None,
            "strain_energy": strain,
        })

strain_df = pd.DataFrame(strain_rows)
display(strain_df.round(3))

# StrainEnergyFilter 適用
print("\nStrainEnergyFilter (threshold=10 kcal/mol) after filtering:")
strain_filter = StrainEnergyFilter(threshold=10.0)
mask = strain_filter.apply(strain_df)
filtered_df = strain_df[mask]
print(f"  {len(strain_df)} → {len(filtered_df)} poses passed strain filter")
display(filtered_df.round(3))

## 9. ウォーヘッドライブラリのスクリーニング
## Warhead Library Screening

**StructuralAlertFilter** を使って、スクリーニングライブラリ中の  
不必要に反応性の高い化合物（PAINS / BRENK 違反）をフィルタリングします。

> 注意: アクリルアミドは PAINS/BRENK でフラグされることがありますが、  
> 共有結合ドッキングが目的の場合は「ウォーヘッドとして意図的に使用」するため  
> BRENK 違反を警告として記録しつつも除去しないアプローチを推奨します。

In [ ]:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.preparation.normalize import standardize_df
from mdatools.docking.analysis.properties import add_properties_to_df

# ダミーライブラリ: アクリルアミド系ウォーヘッド + 対照
warhead_library = pd.DataFrame({
    "compound_id": [
        "AFT-001",   # afatinib
        "AFT-002",   # afatinib analog (shorter chain)
        "OSI-001",   # osimertinib
        "COVA-001",  # generic acrylamide warhead scaffold
        "INERT-001", # non-reactive control
        "PAINS-001", # PAINS compound (should be flagged)
    ],
    "smiles": [
        AFATINIB_SMILES,
        "CC=CC(=O)Nc1cc2c(Nc3ccc(F)cc3)ncnc2cc1OC",
        OSIMERTINIB_SMILES,
        "C=CC(=O)Nc1ccc2ncnc(Nc3cccc(F)c3)c2c1",
        "Cc1ccc2ncnc(Nc3cccc(F)c3)c2c1",       # no warhead
        "O=C(c1ccc(O)cc1)/C=C/c1ccc(O)cc1",    # chalcone (PAINS)
    ],
    "warhead_type": [
        "acrylamide", "acrylamide", "acrylamide",
        "acrylamide", "none", "chalcone",
    ],
})

# 標準化
std_df = standardize_df(warhead_library, smiles_col="smiles")

# 物性追加
std_df = add_properties_to_df(std_df, smiles_col="smiles")

# BRENK フィルタ (mode="annotate" で全件保持しつつフラグを追加)
brenk_filter = StructuralAlertFilter(
    catalog_name="BRENK",
    mode="annotate",
    smiles_col="smiles",
)
brenk_filter.apply(std_df)

display(std_df[["compound_id", "warhead_type", "mw", "logp", "qed",
                "structural_alert", "structural_alert_types"]].round(2))

print("\n注意: アクリルアミドは BRENK でフラグされることがありますが、")
print("共有結合ドッキング目的の場合は除去せず警告として記録することを推奨します。")

## 10. 分子物性と世代間比較
## Molecular Properties and Generation Comparison

In [ ]:
rows = []
for pdb_id, mol in ligand_mols.items():
    props = calculate_properties(mol)
    rows.append({
        "compound": f"{LIGAND_CODES[pdb_id]} ({pdb_id})",
        "generation": "2nd gen (irreversible)" if pdb_id == "4G5J" else "3rd gen (T790M)",
        "warhead": "acrylamide → Cys797",
        **props,
    })

props_df = pd.DataFrame(rows).set_index("compound")
display(props_df.round(3))

### 10.1 結合様式サマリー
### Binding Mode Summary

| 特徴 | Afatinib (4G5J) | Osimertinib (6LUD) |
|------|----------------|--------------------|
| 世代 | **2nd gen** | **3rd gen** |
| ウォーヘッド | Acrylamide | Acrylamide (indole-based) |
| Cys797 共有結合 | ✓ | ✓ |
| T790M 耐性克服 | ✗ (wild-type 選択的) | ✓ (Met790 と疎水接触) |
| HER2 汎阻害 | ✓ (毒性要因) | △ (EGFR 選択的) |
| Met793 ヒンジ H 結合 | ✓ | ✓ |
| 選択性戦略 | 不可逆化で効力↑ | T790M ポケットへの適合 |

**T790M 耐性機序:**  
Thr790→Met790 変異により疎水性が増し、1st/2nd gen の ATP との競合が不利に。  
Osimertinib は Met790 と疎水接触できるよう設計されることで耐性を克服。

## 11. 次のステップ / Next Steps

このノートブックで示したワークフロー:

```
PDB ダウンロード → Cys797 検出 → ウォーヘッド確認
  → 受容体準備 → グリッドボックス (Cys797 包含確認)
  → UniDock2 covalent docking → ProLIF 解析
  → 歪みエネルギー評価 → StructuralAlertFilter
  → 2nd/3rd gen 比較 → 化合物選択
```

**発展的な解析:**
- `validate_poses_posebusters()` — 共有結合ポーズの物理化学的妥当性検証
- `compute_consensus_score()` — 4G5J (WT) と 6LUD (T790M) に対する consensus scoring
  → 両構造への選択性を考慮した化合物選択に活用
- C797S 耐性克服を目指した 4th gen 阻害剤探索への応用

---

**EGFR シリーズの全体像:**
- Issue #75: `06_egfr_kinase.ipynb` — 非共有結合 (erlotinib / lapatinib, DFG-in vs DFG-out)
- Issue #77: `08_egfr_covalent.ipynb` — 共有結合 (afatinib / osimertinib, 2nd/3rd gen) ← このノートブック

**関連 Issue:**
- Issue #78: KRAS G12C 共有結合ドッキング
- Issue #79: Mpro 共有結合ドッキング